In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import mlflow
import mlflow.pytorch

import importlib
import model
importlib.reload(model)
from model import SLRHybridModel


CONFIG = {
    "num_classes": 353,      
    "batch_size": 32,         
    "lr": 5e-4,              
    "weight_decay": 1e-4,    
    "epochs": 50,           
    "patience": 15,         
    "input_dim": 99,
    "hidden_dim": 128,      
    "experiment_name": "SSL400-Hybrid-Final-Optimized-center landmarks",
    "mlflow_tracking_uri": "sqlite:///mlflow.db"
}

def train_hybrid():
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    
    # 1. Load Preprocessed Data
    data_dir = "../../ssl400_processed_paper"
    try:
        train_data = np.load(os.path.join(data_dir, "train_data.npz"))
        val_data = np.load(os.path.join(data_dir, "val_data.npz"))
        
        X_train = torch.from_numpy(train_data["X"]).float()
        y_train = torch.from_numpy(train_data["y"]).long()
        X_val = torch.from_numpy(val_data["X"]).float()
        y_val = torch.from_numpy(val_data["y"]).long()
    except FileNotFoundError:
        print(f"Error: Data files not found in {data_dir}. Run preprocess_paper.py first.")
        return

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=CONFIG["batch_size"])

    
    model_instance = SLRHybridModel(CONFIG["num_classes"], CONFIG["input_dim"], CONFIG["hidden_dim"]).to(device)
    
    
    optimizer = torch.optim.AdamW(model_instance.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.2) 
    
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    
   
    mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
    mlflow.set_experiment(CONFIG["experiment_name"])
    
    with mlflow.start_run():
        mlflow.log_params(CONFIG)
        mlflow.log_param("param_count", sum(p.numel() for p in model_instance.parameters()))

        best_val_acc = 0.0
        early_stop_counter = 0

        print(f"\nStarting training for {CONFIG['num_classes']} classes...")
        for epoch in range(CONFIG["epochs"]):
            
            model_instance.train()
            train_loss, train_correct = 0, 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                optimizer.zero_grad()
                output = model_instance(X_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                
                
                torch.nn.utils.clip_grad_norm_(model_instance.parameters(), max_norm=1.0)
                optimizer.step()
                
                train_loss += loss.item()
                train_correct += (output.argmax(1) == y_batch).type(torch.float).sum().item()

            model_instance.eval()
            val_loss, val_correct = 0, 0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    output = model_instance(X_batch)
                    val_loss += criterion(output, y_batch).item()
                    val_correct += (output.argmax(1) == y_batch).type(torch.float).sum().item()

           
            train_acc = train_correct / len(X_train)
            val_acc = val_correct / len(X_val)
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)

            mlflow.log_metrics({
                "train_loss": avg_train_loss,
                "train_acc": train_acc,
                "val_loss": avg_val_loss,
                "val_acc": val_acc
            }, step=epoch)

            scheduler.step(val_acc)

            print(f"Epoch {epoch:03d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

           
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model_instance.state_dict(), "best_hybrid_model.pt")
                early_stop_counter = 0
                print(f"  --> New Best Val Accuracy: {best_val_acc:.4f} (Saved)")
            else:
                early_stop_counter += 1
            
            if early_stop_counter >= CONFIG["patience"]:
                print(f"\nEarly stopping triggered after {epoch} epochs.")
                break

        
        mlflow.pytorch.log_model(model_instance, "model")
        print(f"\nTraining complete. Best Validation Accuracy: {best_val_acc:.4%}")

if __name__ == "__main__":
    train_hybrid()
    
    

Training on: cpu

Starting training for 353 classes...

Starting training for 353 classes...
Epoch 000 | Train Acc: 0.0557 | Val Acc: 0.1595 | Val Loss: 4.5590
  --> New Best Val Accuracy: 0.1595 (Saved)
Epoch 000 | Train Acc: 0.0557 | Val Acc: 0.1595 | Val Loss: 4.5590
  --> New Best Val Accuracy: 0.1595 (Saved)
Epoch 001 | Train Acc: 0.1105 | Val Acc: 0.2374 | Val Loss: 4.1923
  --> New Best Val Accuracy: 0.2374 (Saved)
Epoch 001 | Train Acc: 0.1105 | Val Acc: 0.2374 | Val Loss: 4.1923
  --> New Best Val Accuracy: 0.2374 (Saved)
Epoch 002 | Train Acc: 0.1556 | Val Acc: 0.2879 | Val Loss: 3.9223
  --> New Best Val Accuracy: 0.2879 (Saved)
Epoch 002 | Train Acc: 0.1556 | Val Acc: 0.2879 | Val Loss: 3.9223
  --> New Best Val Accuracy: 0.2879 (Saved)
Epoch 003 | Train Acc: 0.2080 | Val Acc: 0.3852 | Val Loss: 3.6463
  --> New Best Val Accuracy: 0.3852 (Saved)
Epoch 003 | Train Acc: 0.2080 | Val Acc: 0.3852 | Val Loss: 3.6463
  --> New Best Val Accuracy: 0.3852 (Saved)
Epoch 004 | Train A

2026/04/02 13:04:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 13:04:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/04/02 13:04:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | Train Acc: 0.9646 | Val Acc: 0.7704 | Val Loss: 2.5271

Training complete. Best Validation Accuracy: 78.5992%

Training complete. Best Validation Accuracy: 78.5992%
